In [ ]:
"""
05_market_share_analysis.py
===========================
Compute weekly and overall market share of music labels
based on chart view counts.

INPUT  : india_youtube_labeled.csv
OUTPUT : weekly_label_market_share.csv
         overall_label_market_share.csv
         weekly_label_market_share_no_indie.csv
"""

from __future__ import annotations

import logging
from pathlib import Path

import pandas as pd


# ── Config ────────────────────────────────────────────────────────────────────

INPUT_FILE            = Path("india_youtube_labeled.csv")
WEEKLY_OUTPUT_FILE          = Path("weekly_label_market_share.csv")
OVERALL_OUTPUT_FILE         = Path("overall_label_market_share.csv")
WEEKLY_LABELED_OUTPUT_FILE  = Path("weekly_label_market_share_no_indie.csv")
OVERALL_LABELED_OUTPUT_FILE = Path("overall_label_market_share_no_indie.csv")

REQUIRED_COLUMNS = {"week", "label", "views"}


# ── Logging ───────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ── Helpers ───────────────────────────────────────────────────────────────────

def pct(series: pd.Series, total: pd.Series | int | float) -> pd.Series:
    """Return percentage share rounded to 2 dp, safe against zero division."""
    if total == 0:
        return pd.Series([pd.NA] * len(series), index=series.index)
    return (series / total * 100).round(2)


# ── Core computations ─────────────────────────────────────────────────────────

def compute_weekly_share(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a DataFrame with columns:
        week, label, label_views, total_weekly_views,
        market_share, market_share_pct
    sorted by week asc, market_share_pct desc.
    """
    label_weekly = (
        df.groupby(["week", "label"], sort=True)["views"]
        .sum()
        .rename("label_views")
        .reset_index()
    )

    weekly_totals = (
        label_weekly.groupby("week")["label_views"]
        .sum()
        .rename("total_weekly_views")
    )

    result = label_weekly.join(weekly_totals, on="week")

    result["market_share"]     = result["label_views"] / result["total_weekly_views"].replace(0, pd.NA)
    result["market_share_pct"] = (result["market_share"] * 100).round(2)

    return result.sort_values(
        ["week", "market_share_pct"],
        ascending=[True, False],
    ).reset_index(drop=True)


def compute_overall_share(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a DataFrame with columns:
        label, total_views, market_share_pct
    sorted by total_views desc.
    """
    overall = (
        df.groupby("label")["views"]
        .sum()
        .sort_values(ascending=False)
        .rename("total_views")
        .reset_index()
    )

    overall["market_share_pct"] = pct(overall["total_views"], overall["total_views"].sum())

    return overall


def compute_labeled_share(df: pd.DataFrame, exclude: str = "Independent") -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Recompute weekly and overall market share after dropping *exclude* label.
    Totals are recalculated from scratch so shares sum to 100% within the
    remaining labels — not relative to the full market.

    Returns (weekly_df, overall_df).
    """
    df_filtered = df[df["label"] != exclude].copy()

    weekly_df  = compute_weekly_share(df_filtered)
    overall_df = compute_overall_share(df_filtered)

    return weekly_df, overall_df


# ── Main ──────────────────────────────────────────────────────────────────────

def main() -> None:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

    log.info("Loading %s", INPUT_FILE)
    df = pd.read_csv(INPUT_FILE)
    log.info("Raw shape: %s", df.shape)

    # ── Validate ──────────────────────────────────────────────────────────────
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # ── Clean ─────────────────────────────────────────────────────────────────
    n_before = len(df)
    df["views"] = pd.to_numeric(df["views"], errors="coerce")

    bad_views = df["views"].isna().sum()
    if bad_views:
        log.warning("Dropping %d rows with non-numeric views", bad_views)

    df = df.dropna(subset=["views"])
    df["views"] = df["views"].astype(int)
    log.info("Clean shape: %s  (%d rows dropped)", df.shape, n_before - len(df))

    # ── Weekly share ──────────────────────────────────────────────────────────
    weekly_df = compute_weekly_share(df)
    log.info("Weekly share shape: %s", weekly_df.shape)

    weekly_df.to_csv(WEEKLY_OUTPUT_FILE, index=False)
    log.info("Saved → %s", WEEKLY_OUTPUT_FILE)

    # ── Overall share ─────────────────────────────────────────────────────────
    overall_df = compute_overall_share(df)

    overall_df.to_csv(OVERALL_OUTPUT_FILE, index=False)
    log.info("Saved → %s", OVERALL_OUTPUT_FILE)

    # ── Summary ───────────────────────────────────────────────────────────────
    print("\n── Overall label market share ───────────────────────────────────\n")
    print(overall_df.to_string(index=False))

    print("\n── Weekly share sample (first 30 rows) ──────────────────────────\n")
    print(weekly_df.head(30).to_string(index=False))

    weeks = weekly_df["week"].nunique()
    labels = weekly_df["label"].nunique()
    log.info("Done — %d weeks × %d labels", weeks, labels)

    # ── Labeled-only share (Independent excluded) ──────────────────────────
    weekly_labeled_df, overall_labeled_df = compute_labeled_share(df)

    weekly_labeled_df.to_csv(WEEKLY_LABELED_OUTPUT_FILE, index=False)
    log.info("Saved → %s", WEEKLY_LABELED_OUTPUT_FILE)

    overall_labeled_df.to_csv(OVERALL_LABELED_OUTPUT_FILE, index=False)
    log.info("Saved → %s", OVERALL_LABELED_OUTPUT_FILE)

    print("\n── Overall market share (Independent excluded) ───────────────────\n")
    print(overall_labeled_df.to_string(index=False))


if __name__ == "__main__":
    main()

11:12:35  INFO      Loading india_youtube_labeled.csv
11:12:36  INFO      Raw shape: (28000, 22)
11:12:36  INFO      Clean shape: (28000, 22)  (0 rows dropped)
11:12:36  INFO      Weekly share shape: (3368, 6)
11:12:36  INFO      Saved → weekly_label_market_share.csv
11:12:36  INFO      Saved → overall_label_market_share.csv
11:12:36  INFO      Done — 280 weeks × 21 labels
11:12:36  INFO      Saved → weekly_label_market_share_no_indie.csv
11:12:36  INFO      Saved → overall_label_market_share_no_indie.csv



── Overall label market share ───────────────────────────────────

            label  total_views  market_share_pct
      Independent 101743408873             52.82
         T-Series  34170973922             17.74
         Saregama  11452896791              5.95
             Sony  10129549086              5.26
     Aditya Music   7058233636              3.66
              Zee   5323587498              2.76
              YRF   4389153223              2.28
             Tips   3818340085              1.98
       Wave Music   2608663472              1.35
    Desi Melodies   2590552940              1.34
      Think Music   2410153245              1.25
Worldwide Records   1660378410              0.86
    Speed Records   1088509953              0.57
         Play DMF    977584830              0.51
              Sun    942141154              0.49
       White Hill    798901539              0.41
              MRT    599707249              0.31
      Anand Audio    523365519              0.27
 